# 📊 Model Evaluation — Manufacturing Predictive Maintenance

Detailed evaluation of the trained LightGBM model:
- Confusion matrix
- Feature importance
- Prediction distribution analysis

**Reads from:** `gold_ml_features`, saved model

**Writes to:** `gold_ml_feature_importance`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, when, count
from pyspark.ml.feature import VectorAssembler, StringIndexer
from synapse.ml.lightgbm import LightGBMClassificationModel

spark = SparkSession.builder.getOrCreate()
print('Spark session ready')

In [ ]:
# Load model
model = LightGBMClassificationModel.load('Files/models/predictive_maintenance_lgbm')
print('Model loaded')

In [ ]:
# Prepare test data (same pipeline as training)
features_df = spark.read.table('gold_ml_features')

numeric_features = [
    'avg_temp', 'std_temp', 'max_temp', 'min_temp', 'temp_range',
    'avg_pressure', 'std_pressure', 'max_pressure',
    'avg_vibration', 'std_vibration', 'max_vibration',
    'avg_humidity',
    'reading_count', 'temp_anomaly_count', 'vib_anomaly_count', 'anomaly_ratio',
    'total_units', 'total_defects', 'total_downtime',
    'batch_count', 'avg_yield', 'avg_defect_rate',
    'equipment_age_days',
]

indexer_type = StringIndexer(inputCol='machine_type', outputCol='machine_type_idx', handleInvalid='keep')
indexer_line = StringIndexer(inputCol='production_line', outputCol='production_line_idx', handleInvalid='keep')
indexed_df = indexer_type.fit(features_df).transform(features_df)
indexed_df = indexer_line.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + ['machine_type_idx', 'production_line_idx']
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='skip')
model_df = assembler.transform(indexed_df)

_, test_df = model_df.randomSplit([0.8, 0.2], seed=42)
predictions = model.transform(test_df)
print(f'Test predictions: {predictions.count():,} rows')

In [ ]:
# Confusion matrix
print('=== Confusion Matrix ===')
predictions.groupBy('needs_maintenance', 'prediction').count().orderBy('needs_maintenance', 'prediction').show()

tp = predictions.filter((col('needs_maintenance') == 1) & (col('prediction') == 1)).count()
fp = predictions.filter((col('needs_maintenance') == 0) & (col('prediction') == 1)).count()
fn = predictions.filter((col('needs_maintenance') == 1) & (col('prediction') == 0)).count()
tn = predictions.filter((col('needs_maintenance') == 0) & (col('prediction') == 0)).count()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'TP={tp}, FP={fp}, FN={fn}, TN={tn}')

In [ ]:
# Feature importance
import pandas as pd

try:
    importances = model.getFeatureImportances(importance_type='gain')
    fi_df = pd.DataFrame({
        'feature': all_features,
        'importance': importances[:len(all_features)]
    }).sort_values('importance', ascending=False)
    
    print('=== Top 10 Features by Gain ===')
    for _, row in fi_df.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")
    
    # Save to Gold table
    fi_spark = spark.createDataFrame(fi_df).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')
    print('\nFeature importance saved to gold_ml_feature_importance')
except Exception as e:
    print(f'Could not extract feature importance: {e}')
    # Create placeholder
    fi_data = [(f, 0.0) for f in all_features]
    fi_spark = spark.createDataFrame(fi_data, ['feature', 'importance']).withColumn('model_timestamp', current_timestamp())
    fi_spark.write.mode('overwrite').format('delta').saveAsTable('gold_ml_feature_importance')
    print('Placeholder feature importance saved')

In [ ]:
spark.sql('OPTIMIZE gold_ml_feature_importance')
print('Evaluation complete')